In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import duckdb
import numpy as np
import pandas as pd
from statsbombpy import sb

11 - LaLiga \
7 - Ligue 1 \
2 - Premier League \
12 - Serie A

In [ ]:
def get_teams(comp_id : int, season_id : int = 27):
    df = sb.matches(comp_id, season_id)[["home_team_id", "home_team"]]
    df = df.rename(columns={"home_team_id" : "team_id", "home_team" : "team"})
    df = df.groupby(by = "team_id")
    df = df.first()
    return df

laliga_teams = get_teams(11)
ligue1_teams = get_teams(7)
prem_teams = get_teams(2)
serieA_teams = get_teams(12)

train_laliga = laliga_teams.sample(n=13, random_state=42)
test_laliga = laliga_teams.drop(train_laliga.index)

train_ligue1 =ligue1_teams.sample(n=13, random_state=42)
test_ligue1 = ligue1_teams.drop(train_ligue1.index)

train_prem = prem_teams.sample(n=13, random_state=42)
test_prem = prem_teams.drop(train_prem.index)

train_serieA = serieA_teams.sample(n=13, random_state=42)
test_serieA = serieA_teams.drop(train_serieA.index)

train_teams = pd.concat([train_laliga, train_ligue1, train_prem, train_serieA], ignore_index=True).reset_index()
test_teams = pd.concat([test_laliga, test_ligue1, test_prem, test_serieA], ignore_index=True).reset_index()

In [ ]:
TEST_QUERY = """
-- primary table
CREATE TEMP TABLE pass_distance AS SELECT
    player_id,
    first(player) AS player,
    first(team_id) AS team_id,
    first(team) AS team,
    avg(pass_end_x - x) AS average_pass_x_displacement,
    avg(abs(pass_end_y - y)) AS average_pass_norm_y_displacement,
    count(*) AS pass_total
FROM events
WHERE
    (
        (type = 'Pass') AND
        (position <> 'Goalkeeper')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE carry_distance AS SELECT
    player_id,
    first(player) AS player,
    avg(end_x - x) AS average_carry_x_displacement,
    avg(abs(end_y - y)) AS average_carry_norm_y_displacement,
    count(*) AS carry_total
FROM events
WHERE
    (
        (type = 'Complete Carry')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE player_rel AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    CASE
        WHEN type = 'Complete Carry' THEN end_x
        ELSE x
        END AS x,
    CASE
        WHEN type = 'Complete Carry' THEN end_y
        ELSE y
        END AS y
FROM
    events
WHERE
    (
        (type = 'Pass') OR
        (type = 'Shot') OR
        ((type = 'Complete Carry') AND (carry_end_outcome = 'Other'))
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rel"
CREATE TEMP TABLE player_rel_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rel_x,
    avg(y) AS avg_rel_y,
    avg(40 - ABS(y - 40)) AS avg_normrel_y,
    count(*) AS rel_total
FROM player_rel
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE player_rec AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    x,
    y,
FROM
    events
WHERE 
    (
        (type = 'Ball Receipt*') OR
        (type = 'Ball Recovery') OR
        ((type = '50/50') AND ("50_50".outcome.name = 'Won')) OR
        ((type = 'Duel') AND (duel_outcome = 'Won')) OR
        ((type = 'Interception') AND (interception_outcome = 'Won')) OR
        ((type = 'Pass') AND (pass_type = 'Recovery')) OR
        ((type = 'Shot') AND (shot_type = 'Recovery'))
    )
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rec"
CREATE TEMP TABLE player_rec_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rec_x,
    avg(y) AS avg_rec_y,
    avg(40 - ABS(y - 40)) AS avg_normrec_y,
    count(*) AS rec_total
FROM player_rec
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE key_pass AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS key_pass_total
FROM
    events
WHERE
    (
        (type = 'Pass') AND
        (pass_shot_assist)
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE shot_count AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS shot_total
FROM events
WHERE type = 'Shot' AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE reception AS
    SELECT
        player_id,
        match_id,
        team_id,
        player,
        type,
        match_seconds,
        play_pattern,
        x,
        y,
    FROM
        events
    WHERE
        (
            (type = 'Ball Receipt*') OR
            (type = 'Ball Recovery') OR
            ((type = '50/50') AND ("50_50".outcome.name = 'Won')) OR
            ((type = 'Duel') AND (duel_outcome = 'Won')) OR
            ((type = 'Interception') AND (interception_outcome = 'Won'))
        )
        AND (matchweek >= $1 AND matchweek <= $2)
    ORDER BY
        match_id, team_id, match_seconds;

-- primary table
CREATE TEMP TABLE outcome AS
    SELECT
        player_id,
        match_id,
        team_id,
        type,
        match_seconds
    FROM
        events
    WHERE
        (
            (type = 'Pass') OR
            (type = 'Shot') OR
            (type = 'Complete Carry')
        )
        AND (matchweek >= $1 AND matchweek <= $2)
    ORDER BY
        match_id, team_id, match_seconds;

-- secondary table from "reception" and "outcome"
CREATE TEMP TABLE reception_outcomes AS
SELECT
    l.player_id as player_id,
    l.match_id as match_id,
    l.team_id as team_id,
    l.player as player,
    l.type as type,
    l.match_seconds as match_seconds,
    l.play_pattern as play_pattern,
    l.x as x,
    l.y as y,
    r.type as reception_outcome
FROM reception l LEFT JOIN outcome r ON
    l.player_id = r.player_id AND
    l.match_id = r.match_id AND
    l.team_id = r.team_id AND
    l.match_seconds = r.match_seconds;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_pass_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Pass'
GROUP BY player_id;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_carry_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Complete Carry'
GROUP BY player_id;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_shot_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Shot'
GROUP BY player_id;

-- returned table
SELECT
    l.player_id AS player_id,
    l.player AS player,
    l.team_id AS team_id,
    l.team AS team,
    l.average_pass_x_displacement AS average_pass_x_displacement,
    l.average_pass_norm_y_displacement AS average_pass_norm_y_displacement,
    l.pass_total AS pass_total,
    r.average_carry_x_displacement AS average_carry_x_displacement,
    r.average_carry_norm_y_displacement AS average_carry_norm_y_displacement,
    r.carry_total AS carry_total,
    a.avg_rel_x AS average_release_x,
    a.avg_rel_y AS average_release_y,
    a.avg_normrel_y AS average_norm_release_y,
    a.rel_total AS rel_total,
    b.avg_rec_x AS average_reception_x,
    b.avg_rec_y AS average_reception_y,
    b.avg_normrec_y AS average_norm_reception_y,
    b.rec_total AS rec_total,
    (c.key_pass_total / (c.key_pass_total + d.shot_total)) AS key_pass_shot_ratio,
    c.key_pass_total AS key_pass_total,
    d.shot_total AS shot_total,
    (oc.outcome_count/(op.outcome_count + oc.outcome_count + osh.outcome_count)) AS carry_shotpass_ratio,
    op.outcome_count AS outcome_pass_total,
    oc.outcome_count AS outcome_carry_total,
    osh.outcome_count AS outcome_shot_total,
FROM pass_distance l
INNER JOIN carry_distance r ON
    l.player_id = r.player_id AND
    l.player = r.player
INNER JOIN player_rel_avg a ON
    l.player_id = a.player_id AND
    l.player = a.player
INNER JOIN player_rec_avg b ON
    l.player_id = b.player_id AND
    l.player = b.player
INNER JOIN key_pass c ON
    l.player_id = c.player_id AND
    l.player = c.player
INNER JOIN shot_count d ON
    l.player_id = d.player_id AND
    l.player = d.player
INNER JOIN outcome_pass_count op ON
    l.player_id = op.player_id AND
    l.player = op.player
INNER JOIN outcome_carry_count oc ON
    l.player_id = oc.player_id AND
    l.player = oc.player
INNER JOIN outcome_shot_count osh ON
    l.player_id = osh.player_id AND
    l.player = osh.player
WHERE
    (a.rel_total >= $1 AND
    b.rec_total >= $1);
"""

In [ ]:
statements = [s.strip() for s in TEST_QUERY.split(';') if s.strip()]

setup_statements = statements[:-1]
final_query = statements[-1]

PRIMARY_TABLES = [0,1,2,4,6,7,8,9]
SECONDARY_TABLES = [3,5,10]
TERTIARY_TABLES = [11,12,13]

DB_PATH = '../../data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
for x in PRIMARY_TABLES:
    con.execute(setup_statements[x], [0,40])
for y in SECONDARY_TABLES:
    con.execute(setup_statements[y])
for z in TERTIARY_TABLES:
    con.execute(setup_statements[z])
main_df = con.execute(final_query, [50]).df()
con.close()

In [ ]:
TEAMS_PLAYED_FOR = """
SELECT player_id, count(DISTINCT team_id) AS teams_played_for, first(team_id) AS team_id, last(team_id) AS second_team_id FROM events GROUP BY player_id;
"""
DB_PATH = '../../data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
tpf_df = con.execute(TEAMS_PLAYED_FOR).df()
con.close()

exclude_players = tpf_df[tpf_df["teams_played_for"] > 1][["player_id", "team_id"]]
exc_df = main_df[~(main_df["player_id"].isin(exclude_players["player_id"]))]

train_df = exc_df[(exc_df["team_id"].isin(train_teams["index"]))]
train_pid = pd.merge(train_df["player_id"], tpf_df[["player_id", "team_id"]], how = "left", on = "player_id")
train_pid["player_id"] = train_pid["player_id"].astype(int)
train_pid = train_pid.sort_values(by = "team_id").reset_index(drop=True)

test_df = exc_df[(exc_df["team_id"].isin(test_teams["index"]))]
test_pid = pd.merge(test_df["player_id"], tpf_df[["player_id", "team_id"]], how = "left", on = "player_id")
test_pid["player_id"] = test_pid["player_id"].astype(int)
test_pid = test_pid.sort_values(by = "team_id").reset_index(drop=True)


In [ ]:
train_player_ids = train_pid.copy()
train_merged = pd.merge(left = train_player_ids, right = train_player_ids, how = "inner", on = "team_id")

train_player_id_pairs = train_merged[~(train_merged["player_id_x"] == train_merged["player_id_y"])]
train_pairs_df = train_player_id_pairs.copy()

con = duckdb.connect(DB_PATH, read_only=True)
con.register("pairs_df", train_pairs_df)

VECTORIZED_QUERY = """
WITH shot_events AS (
    -- 1. Pre-filter only Shot events within matchweeks 0 to 24
    SELECT 
        player_id AS shooter_id,
        teamsheet
    FROM events
    WHERE type = 'Shot'
),
pair_metrics AS (
    -- 2. Join 9,000 pairs against shot events where both players share the pitch
    SELECT 
        p.player_id_x,
        p.player_id_y,
        -- Shots taken by target player (player_id_x) while co-present
        COUNT(CASE WHEN s.shooter_id = p.player_id_x THEN 1 END) AS shot_count,
        -- Total shots taken while both were co-present on pitch
        COUNT(*) AS shot_involvement
    FROM pairs_df p
    JOIN shot_events s
       ON list_contains(s.teamsheet, p.player_id_x)
      AND list_contains(s.teamsheet, p.player_id_y)
    GROUP BY p.player_id_x, p.player_id_y
)
-- 3. Preserve all 9,000 pairs with 0 fallback
SELECT 
    p.player_id_x,
    p.player_id_y,
    COALESCE(m.shot_count, 0) AS shot_count,
    COALESCE(m.shot_involvement, 0) AS shot_involvement,
    CASE 
        WHEN COALESCE(m.shot_involvement, 0) > 0 
        THEN COALESCE(m.shot_count, 0) * 1.0 / m.shot_involvement 
        ELSE 0.0 
    END AS shot_usage
FROM pairs_df p
LEFT JOIN pair_metrics m 
    ON p.player_id_x = m.player_id_x 
   AND p.player_id_y = m.player_id_y;
"""

df_results = con.execute(VECTORIZED_QUERY).df()
con.close()

# Merge results back into your main pairs DataFrame
train_pairs_df = train_pairs_df.merge(
    df_results, on=["player_id_x", "player_id_y"], how="left"
)

In [ ]:
test_player_ids = test_pid.copy()
test_merged = pd.merge(left = test_player_ids, right = test_player_ids, how = "inner", on = "team_id")

test_player_id_pairs = test_merged[~(test_merged["player_id_x"] == test_merged["player_id_y"])]
test_pairs_df = test_player_id_pairs.copy()

con = duckdb.connect(DB_PATH, read_only=True)
con.register("pairs_df", test_pairs_df)

VECTORIZED_QUERY = """
WITH shot_events AS (
    -- 1. Pre-filter only Shot events within matchweeks 0 to 24
    SELECT 
        player_id AS shooter_id,
        teamsheet
    FROM events
    WHERE type = 'Shot'
),
pair_metrics AS (
    -- 2. Join 9,000 pairs against shot events where both players share the pitch
    SELECT 
        p.player_id_x,
        p.player_id_y,
        -- Shots taken by target player (player_id_x) while co-present
        COUNT(CASE WHEN s.shooter_id = p.player_id_x THEN 1 END) AS shot_count,
        -- Total shots taken while both were co-present on pitch
        COUNT(*) AS shot_involvement
    FROM pairs_df p
    JOIN shot_events s
       ON list_contains(s.teamsheet, p.player_id_x)
      AND list_contains(s.teamsheet, p.player_id_y)
    GROUP BY p.player_id_x, p.player_id_y
)
-- 3. Preserve all 9,000 pairs with 0 fallback
SELECT 
    p.player_id_x,
    p.player_id_y,
    COALESCE(m.shot_count, 0) AS shot_count,
    COALESCE(m.shot_involvement, 0) AS shot_involvement,
    CASE 
        WHEN COALESCE(m.shot_involvement, 0) > 0 
        THEN COALESCE(m.shot_count, 0) * 1.0 / m.shot_involvement 
        ELSE 0.0 
    END AS shot_usage
FROM pairs_df p
LEFT JOIN pair_metrics m 
    ON p.player_id_x = m.player_id_x 
   AND p.player_id_y = m.player_id_y;
"""

df_results = con.execute(VECTORIZED_QUERY).df()
con.close()

# Merge results back into your main pairs DataFrame
test_pairs_df = test_pairs_df.merge(
    df_results, on=["player_id_x", "player_id_y"], how="left"
)

In [ ]:
print(len(train_pairs_df))
print(len(test_pairs_df))

In [ ]:
cols = [
    "player_id",
    "average_pass_x_displacement",
    "average_pass_norm_y_displacement",
    "average_carry_x_displacement",
    "average_carry_norm_y_displacement",
    "average_release_x",
    "average_norm_release_y",
    "average_reception_x",
    "average_norm_reception_y",
    "key_pass_shot_ratio",
    "carry_shotpass_ratio"
]

features_train = train_df[cols].copy()
features_train["player_id"] = features_train["player_id"].astype(int)

features_test = test_df[cols].copy()
features_test["player_id"] = features_test["player_id"].astype(int)

train_pairs_df = train_pairs_df.merge(features_train.add_suffix("_1"), left_on="player_id_x",
    right_on="player_id_1",
    how="left",
)

train_pairs_df = train_pairs_df.merge(
    features_train.add_suffix("_2"),
    left_on="player_id_y",
    right_on="player_id_2",
    how="left",
)

test_pairs_df = test_pairs_df.merge(features_train.add_suffix("_1"), left_on="player_id_x",
    right_on="player_id_1",
    how="left",
)

test_pairs_df = test_pairs_df.merge(
    features_train.add_suffix("_2"),
    left_on="player_id_y",
    right_on="player_id_2",
    how="left",
)

In [ ]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset
import torch

p_features = [
    "average_pass_x_displacement_1",
    "average_pass_norm_y_displacement_1",
    "average_carry_x_displacement_1",
    "average_carry_norm_y_displacement_1",
    "average_release_x_1",
    "average_norm_release_y_1",
    "average_reception_x_1",
    "average_norm_reception_y_1",
    "key_pass_shot_ratio_1",
    "carry_shotpass_ratio_1",
    "average_pass_x_displacement_2",
    "average_pass_norm_y_displacement_2",
    "average_carry_x_displacement_2",
    "average_carry_norm_y_displacement_2",
    "average_release_x_2",
    "average_norm_release_y_2",
    "average_reception_x_2",
    "average_norm_reception_y_2",
    "key_pass_shot_ratio_2",
    "carry_shotpass_ratio_2"
]

target_col = ["shot_usage"]

train_X = train_pairs_df[p_features].copy()
train_Y = train_pairs_df["shot_usage"].copy()

test_X = test_pairs_df[p_features].copy()
test_Y = test_pairs_df["shot_usage"].copy()

scaler = StandardScaler()
train_X = scaler.fit_transform(train_X)
test_X = scaler.transform(test_X)

class PlayerPairDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets.values, dtype=torch.float32)
 
    def __len__(self):
        return len(self.X)
 
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = PlayerPairDataset(train_X, train_Y)
test_dataset = PlayerPairDataset(test_X, test_Y)

In [ ]:
from torch.utils.data import DataLoader
from torch import nn, optim

device = torch.device("cpu")

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64)

loss_fn = nn.MSELoss()

shotusageModel = nn.Sequential(
    nn.Linear(20, 14),
    nn.ReLU(),
    nn.Linear(14, 1)
)
shotusageModel.to(device)

# 2. Define the optimizer
optimizer = optim.Adam(shotusageModel.parameters(), lr=1e-3)

# 4. Training loop
epochs = 20

for epoch in range(epochs):
    shotusageModel.train()  # Set model to training mode
    running_loss = 0.0

    for inputs, targets in train_dataloader:
        # Move data to device
        inputs, targets = inputs.to(device), targets.to(device)

        # Ensure targets have the correct shape (batch_size, 1) for MSE Loss
        if targets.ndim == 1:
            targets = targets.unsqueeze(1)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = shotusageModel(inputs)
        loss = loss_fn(outputs, targets.float())

        # Backward pass & update weights
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataloader.dataset)
    print(f"Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss:.4f}")

# 1. Set model to evaluation mode
shotusageModel.eval()

test_loss = 0.0

# 2. Disable gradient computation for evaluation
with torch.no_grad():
    for inputs, targets in test_dataloader:
        inputs = inputs.to(device).float()
        targets = targets.to(device).float()

        if targets.ndim == 1:
            targets = targets.unsqueeze(1)

        # Forward pass only
        outputs = shotusageModel(inputs)
        loss = loss_fn(outputs, targets)

        # Accumulate loss (batch loss * batch size)
        test_loss += loss.item() * inputs.size(0)

# 3. Calculate average loss over the entire test set
avg_test_loss = test_loss / len(test_dataloader.dataset)
print(f"Test MSE Loss: {avg_test_loss:.4f}")
print(f"Test RMSE Loss: {avg_test_loss ** 0.5:.4f}")
